# VR Cybersickness Prediction — Dual-Task Baseline Framework

> **ACM Multimedia (MM) Dataset Track**

This notebook establishes a **dual-task prediction framework** for the multimodal VR cybersickness dataset:

- **Task 1**: Continuous Cybersickness Tracking (Minute-Level) — predict per-minute score (0-8) and sick (binary)
- **Task 2**: Session-Level Cybersickness Prediction — predict SSQ Total Score (0-209)

---

In [ ]:
import platform
import torch
print('OS:', platform.system())
print('CUDA available:', torch.cuda.is_available())

## Global Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, f1_score, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"    GPU: {torch.cuda.get_device_name(0)}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SICK_THRESHOLD = 4.0
print(f"[✓] SICK_THRESHOLD = {SICK_THRESHOLD}")

# ==========================================
# Task 1: Continuous Cybersickness Tracking (Minute-Level)
# ==========================================

**Target**: score (0-8) + sick (binary)  
**Models**: XGBoost, GRU, Multimodal Fusion

## Task 1 Data Loading

In [ ]:
# Load data
df = pd.read_excel('../sample_data/sample_task1.xlsx')

# Generate binary label
df['sick'] = (df['score'] >= SICK_THRESHOLD).astype(int)

# Feature column definitions
ECG_COLS = ['HR', 'SDNN', 'RMSSD']
EDA_COLS = ['SCL', 'SCR_AMP', 'SCR_COUNT']
RESP_COLS = ['RESP_RATE', 'RESP_AMP']
EYE_COLS = ['Blink_Frequency', 'Mean_Blink_Duration', 'PERCLOS_Proxy', 'Tracking_Loss_Ratio']
FEAT_COLS = ECG_COLS + EDA_COLS + RESP_COLS + EYE_COLS
VALID_COLS = ['ecg_valid', 'eda_valid', 'resp_valid', 'eye_valid']

PARTICIPANTS = sorted(df['participant'].unique())
print(f'[Task 1] Loaded {len(df)} rows, {len(PARTICIPANTS)} participants')
print(f'         sick rate: {df["sick"].mean()*100:.1f}%')

## Task 1 Utility Functions

In [ ]:
def loso_split(df, test_p):
    return df[df['participant'] != test_p].copy(), df[df['participant'] == test_p].copy()

def fit_transform_X(X_tr, X_te):
    imp = SimpleImputer(strategy='mean')
    sc = StandardScaler()
    return sc.fit_transform(imp.fit_transform(X_tr)), sc.transform(imp.transform(X_te))

def eval_regression(y_true, y_pred):
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'r': pearsonr(y_true, y_pred)[0] if len(np.unique(y_pred)) > 1 else 0.0
    }

def eval_classification(y_true, y_pred, y_prob=None):
    return {
        'Acc': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'AUC': roc_auc_score(y_true, y_prob) if y_prob is not None and len(np.unique(y_true)) > 1 else 0.0
    }

print("[✓] Task 1 utils defined")

## Task 1 Model Definitions

In [ ]:
# GRU model
class GRUPredictor(nn.Module):
    def __init__(self, input_size, hidden=64, task='reg'):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden, 2, batch_first=True, dropout=0.3)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1))
    
    def forward(self, x):
        _, h_n = self.gru(x)
        return self.head(h_n[-1]).squeeze(-1)

# Multimodal Fusion model (enhanced regularization: BatchNorm1d + Dropout=0.5)
class ModalityGate(nn.Module):
    def __init__(self, in_dim, out_dim=16):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim), nn.ReLU())
    
    def forward(self, x, valid):
        h = self.net(x)
        return h * valid.unsqueeze(-1)

class MultimodalFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.ecg_gate = ModalityGate(3, 16)
        self.eda_gate = ModalityGate(3, 16)
        self.resp_gate = ModalityGate(2, 16)
        self.eye_gate = ModalityGate(4, 16)
        self.physio_proj = nn.Sequential(nn.Linear(48, 32), nn.LayerNorm(32), nn.ReLU())
        self.eye_proj = nn.Sequential(nn.Linear(16, 16), nn.LayerNorm(16), nn.ReLU())
        self.head = nn.Sequential(
            nn.Linear(48, 32), 
            nn.BatchNorm1d(32), 
            nn.ReLU(), 
            nn.Dropout(0.5), 
            nn.Linear(32, 1)
        )
    
    def forward(self, x_ecg, x_eda, x_resp, x_eye, ecg_v, eda_v, resp_v, eye_v):
        h_p = self.physio_proj(torch.cat([
            self.ecg_gate(x_ecg, ecg_v),
            self.eda_gate(x_eda, eda_v),
            self.resp_gate(x_resp, resp_v)
        ], dim=-1))
        h_e = self.eye_proj(self.eye_gate(x_eye, eye_v))
        return self.head(torch.cat([h_p, h_e], dim=-1)).squeeze(-1)

print("[ok] Task 1 models defined")

In [ ]:
# Training function with early stopping and loss history
# Validation set uses the last 15% of training data (sequential split to prevent leakage)
def train_pytorch_once(model, X_tr, y_tr, task='reg', max_epochs=100, patience=10, 
                       batch_size=128, lr=1e-3, static_tr=None, mask_tr=None):
    n_val = max(8, int(len(X_tr) * 0.15))
    X_sub, X_val = X_tr[:-n_val], X_tr[-n_val:]
    y_sub, y_val = y_tr[:-n_val], y_tr[-n_val:]
    
    if static_tr is not None and mask_tr is not None:
        stat_sub, stat_val = static_tr[:-n_val], static_tr[-n_val:]
        mask_sub, mask_val = mask_tr[:-n_val], mask_tr[-n_val:]
        ds = TensorDataset(torch.tensor(X_sub).to(DEVICE), torch.tensor(stat_sub).to(DEVICE),
                          torch.tensor(mask_sub).to(DEVICE), torch.tensor(y_sub).to(DEVICE))
        val_inputs = [torch.tensor(X_val).to(DEVICE), torch.tensor(stat_val).to(DEVICE), 
                     torch.tensor(mask_val).to(DEVICE)]
    else:
        ds = TensorDataset(torch.tensor(X_sub).to(DEVICE), torch.tensor(y_sub).to(DEVICE))
        val_inputs = [torch.tensor(X_val).to(DEVICE)]
    
    y_val_t = torch.tensor(y_val).to(DEVICE)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.MSELoss() if task == 'reg' else nn.BCEWithLogitsLoss()
    
    best_val, best_wts, pat_cnt = float('inf'), copy.deepcopy(model.state_dict()), 0
    history = {'train_loss': [], 'val_loss': []}
    
    for ep in range(1, max_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for batch in loader:
            opt.zero_grad()
            inputs, yb = batch[:-1], batch[-1]
            loss = crit(model(*inputs), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item()
        
        model.eval()
        with torch.no_grad():
            val_loss = crit(model(*val_inputs), y_val_t).item()
        
        history['train_loss'].append(epoch_loss / len(loader))
        history['val_loss'].append(val_loss)
        
        if val_loss < best_val - 1e-4:
            best_val, best_wts, pat_cnt = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            pat_cnt += 1
            if pat_cnt >= patience: break
    
    model.load_state_dict(best_wts)
    return model, history

print("[ok] Training function defined")

## Task 1 Training & Evaluation

In [ ]:
# XGBoost LOSO-CV
def run_xgb_task1():
    y_true_all, y_pred_all = [], []
    last_model = None
    for test_p in tqdm(PARTICIPANTS, desc='XGBoost'):
        tr, te = loso_split(df, test_p)
        X_tr, X_te = fit_transform_X(tr[FEAT_COLS].values, te[FEAT_COLS].values)
        y_tr, y_te = tr['score'].values, te['score'].values
        
        model = xgb.XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=SEED)
        model.fit(X_tr, y_tr)
        y_pred_all.extend(model.predict(X_te))
        y_true_all.extend(y_te)
        last_model = model
    
    y_true_all, y_pred_all = np.array(y_true_all), np.array(y_pred_all)
    metrics = eval_regression(y_true_all, y_pred_all)
    return metrics, y_true_all, y_pred_all, last_model

xgb_result, xgb_y_true, xgb_y_pred, xgb_model = run_xgb_task1()
print(f"[XGBoost] RMSE={xgb_result['RMSE']:.3f}, MAE={xgb_result['MAE']:.3f}, r={xgb_result['r']:.3f}")

In [ ]:
# Linear Regression LOSO-CV (naive baseline)
from sklearn.linear_model import LinearRegression

def run_lr_task1():
    y_true_all, y_pred_all = [], []
    all_coefs = []  # store per-fold coefficients
    
    for test_p in tqdm(PARTICIPANTS, desc='LR'):
        tr, te = loso_split(df, test_p)
        X_tr, y_tr = tr[FEAT_COLS].values, tr['score'].values
        X_te, y_te = te[FEAT_COLS].values, te['score'].values
        
        X_tr, X_te = fit_transform_X(X_tr, X_te)
        
        model = LinearRegression()
        model.fit(X_tr, y_tr)
        
        all_coefs.append(np.abs(model.coef_))
        
        y_pred_all.extend(model.predict(X_te))
        y_true_all.extend(y_te)
    
    y_true_all, y_pred_all = np.array(y_true_all), np.array(y_pred_all)
    metrics = eval_regression(y_true_all, y_pred_all)
    
    avg_coefs = np.mean(all_coefs, axis=0)  # average coefficients across folds
    
    return metrics, y_true_all, y_pred_all, avg_coefs

lr_result, lr_y_true, lr_y_pred, lr_coefs = run_lr_task1()
print(f"[LR] RMSE={lr_result['RMSE']:.3f}, MAE={lr_result['MAE']:.3f}, r={lr_result['r']:.3f}")

In [ ]:
# GRU LOSO-CV (sliding window)
def make_windows(df_sub, window=5):
    Xs, ys = [], []
    for (p, o), grp in df_sub.groupby(['participant', 'order']):
        grp = grp.sort_values('time')
        X, y = grp[FEAT_COLS].values, grp['score'].values
        for t in range(window-1, len(X)):
            Xs.append(X[t-window+1:t+1])
            ys.append(y[t])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

def run_gru_task1():
    y_true_all, y_pred_all = [], []
    last_history = None
    for test_p in tqdm(PARTICIPANTS, desc='GRU'):
        tr, te = loso_split(df, test_p)
        X_tr, y_tr = make_windows(tr)
        X_te, y_te = make_windows(te)
        if len(X_te) == 0: continue
        
        X_tr, X_te = fit_transform_X(X_tr.reshape(-1, 12), X_te.reshape(-1, 12))
        X_tr, X_te = X_tr.reshape(-1, 5, 12), X_te.reshape(-1, 5, 12)
        
        model = GRUPredictor(12).to(DEVICE)
        model, last_history = train_pytorch_once(model, X_tr, y_tr, task='reg', max_epochs=50, patience=10, lr=5e-4)
        
        model.eval()
        with torch.no_grad():
            y_pred_all.extend(model(torch.tensor(X_te).to(DEVICE)).cpu().numpy())
        y_true_all.extend(y_te)
    
    y_true_all, y_pred_all = np.array(y_true_all), np.array(y_pred_all)
    metrics = eval_regression(y_true_all, y_pred_all)
    return metrics, y_true_all, y_pred_all, last_history

gru_result, gru_y_true, gru_y_pred, gru_history = run_gru_task1()
print(f"[GRU] RMSE={gru_result['RMSE']:.3f}, MAE={gru_result['MAE']:.3f}, r={gru_result['r']:.3f}")

In [ ]:
# Multimodal Fusion LOSO-CV
def extract_modality_arrays(df_split):
    return (df_split[ECG_COLS].values, df_split[EDA_COLS].values, 
            df_split[RESP_COLS].values, df_split[EYE_COLS].values,
            df_split['ecg_valid'].values, df_split['eda_valid'].values,
            df_split['resp_valid'].values, df_split['eye_valid'].values)

def run_fusion_task1():
    y_true_all, y_pred_all = [], []
    last_history = None
    for test_p in tqdm(PARTICIPANTS, desc='Fusion'):
        tr, te = loso_split(df, test_p)
        arrs_tr = extract_modality_arrays(tr)
        arrs_te = extract_modality_arrays(te)
        
        # Impute + scale each modality independently
        xecg_tr, xecg_te = fit_transform_X(arrs_tr[0], arrs_te[0])
        xeda_tr, xeda_te = fit_transform_X(arrs_tr[1], arrs_te[1])
        xrsp_tr, xrsp_te = fit_transform_X(arrs_tr[2], arrs_te[2])
        xeye_tr, xeye_te = fit_transform_X(arrs_tr[3], arrs_te[3])
        
        y_tr, y_te = tr['score'].values, te['score'].values
        
        # Sequential validation split (last 15%) to prevent temporal leakage
        n_val = max(8, int(len(xecg_tr) * 0.15))
        xecg_s, xecg_v = xecg_tr[:-n_val], xecg_tr[-n_val:]
        xeda_s, xeda_v = xeda_tr[:-n_val], xeda_tr[-n_val:]
        xrsp_s, xrsp_v = xrsp_tr[:-n_val], xrsp_tr[-n_val:]
        xeye_s, xeye_v = xeye_tr[:-n_val], xeye_tr[-n_val:]
        v_ecg_s, v_ecg_v = arrs_tr[4][:-n_val], arrs_tr[4][-n_val:]
        v_eda_s, v_eda_v = arrs_tr[5][:-n_val], arrs_tr[5][-n_val:]
        v_rsp_s, v_rsp_v = arrs_tr[6][:-n_val], arrs_tr[6][-n_val:]
        v_eye_s, v_eye_v = arrs_tr[7][:-n_val], arrs_tr[7][-n_val:]
        y_s, y_v = y_tr[:-n_val], y_tr[-n_val:]
        
        model = MultimodalFusion().to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-2)
        
        # Build 8-input DataLoader
        ds = TensorDataset(
            torch.tensor(xecg_s, dtype=torch.float32).to(DEVICE),
            torch.tensor(xeda_s, dtype=torch.float32).to(DEVICE),
            torch.tensor(xrsp_s, dtype=torch.float32).to(DEVICE),
            torch.tensor(xeye_s, dtype=torch.float32).to(DEVICE),
            torch.tensor(v_ecg_s, dtype=torch.float32).to(DEVICE),
            torch.tensor(v_eda_s, dtype=torch.float32).to(DEVICE),
            torch.tensor(v_rsp_s, dtype=torch.float32).to(DEVICE),
            torch.tensor(v_eye_s, dtype=torch.float32).to(DEVICE),
            torch.tensor(y_s, dtype=torch.float32).to(DEVICE)
        )
        loader = DataLoader(ds, batch_size=128, shuffle=True)
        
        best_val, best_wts, pat_cnt = float('inf'), copy.deepcopy(model.state_dict()), 0
        last_history = {'train_loss': [], 'val_loss': []}
        
        for ep in range(1, 51):
            model.train()
            epoch_loss = 0.0
            for batch in loader:
                inputs, yb = batch[:-1], batch[-1]
                opt.zero_grad()
                loss = nn.MSELoss()(model(*inputs), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                epoch_loss += loss.item()
            
            model.eval()
            with torch.no_grad():
                val_loss = nn.MSELoss()(
                    model(torch.tensor(xecg_v, dtype=torch.float32).to(DEVICE),
                          torch.tensor(xeda_v, dtype=torch.float32).to(DEVICE),
                          torch.tensor(xrsp_v, dtype=torch.float32).to(DEVICE),
                          torch.tensor(xeye_v, dtype=torch.float32).to(DEVICE),
                          torch.tensor(v_ecg_v, dtype=torch.float32).to(DEVICE),
                          torch.tensor(v_eda_v, dtype=torch.float32).to(DEVICE),
                          torch.tensor(v_rsp_v, dtype=torch.float32).to(DEVICE),
                          torch.tensor(v_eye_v, dtype=torch.float32).to(DEVICE)),
                    torch.tensor(y_v, dtype=torch.float32).to(DEVICE)).item()
            
            last_history['train_loss'].append(epoch_loss / len(loader))
            last_history['val_loss'].append(val_loss)
            
            if val_loss < best_val - 1e-4:
                best_val, best_wts, pat_cnt = val_loss, copy.deepcopy(model.state_dict()), 0
            else:
                pat_cnt += 1
                if pat_cnt >= 10: break
        
        model.load_state_dict(best_wts)
        model.eval()
        with torch.no_grad():
            te_pred = model(
                torch.tensor(xecg_te, dtype=torch.float32).to(DEVICE),
                torch.tensor(xeda_te, dtype=torch.float32).to(DEVICE),
                torch.tensor(xrsp_te, dtype=torch.float32).to(DEVICE),
                torch.tensor(xeye_te, dtype=torch.float32).to(DEVICE),
                torch.tensor(arrs_te[4], dtype=torch.float32).to(DEVICE),
                torch.tensor(arrs_te[5], dtype=torch.float32).to(DEVICE),
                torch.tensor(arrs_te[6], dtype=torch.float32).to(DEVICE),
                torch.tensor(arrs_te[7], dtype=torch.float32).to(DEVICE)
            ).cpu().numpy()
        
        y_pred_all.extend(te_pred)
        y_true_all.extend(y_te)
    
    y_true_all, y_pred_all = np.array(y_true_all), np.array(y_pred_all)
    metrics = eval_regression(y_true_all, y_pred_all)
    return metrics, y_true_all, y_pred_all, last_history

fusion_result, fusion_y_true, fusion_y_pred, fusion_history = run_fusion_task1()
print(f"[Fusion] RMSE={fusion_result['RMSE']:.3f}, MAE={fusion_result['MAE']:.3f}, r={fusion_result['r']:.3f}")

In [ ]:
# Task 1 results summary
task1_results = pd.DataFrame({
    'Model': ['Linear Regression', 'XGBoost', 'GRU', 'Multimodal Fusion'],
    'RMSE': [lr_result['RMSE'], xgb_result['RMSE'], gru_result['RMSE'], fusion_result['RMSE']],
    'MAE': [lr_result['MAE'], xgb_result['MAE'], gru_result['MAE'], fusion_result['MAE']],
    'r': [lr_result['r'], xgb_result['r'], gru_result['r'], fusion_result['r']]
})
print("\n[Task 1: Continuous Tracking - Regression Results]")
print(task1_results.to_string(index=False))

In [ ]:
# Visualization utilities (publication-quality)
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['font.family'] = 'sans-serif'

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
def plot_results(y_true, y_pred, model_name, task_name, history=None):
    fig = plt.figure(figsize=(12, 5))
    
    # Prediction scatter plot
    ax1 = plt.subplot(1, 2 if history else 1, 1)
    sns.regplot(x=y_true, y=y_pred, ax=ax1, 
                scatter_kws={'alpha':0.5, 's':20, 'color':'#2b8cbe'}, 
                line_kws={'color':'#de2d26', 'linewidth':2})
    min_val, max_val = min(y_true), max(y_true)
    ax1.plot([min_val, max_val], [min_val, max_val], 'k--', lw=1.5, label='Perfect Fit')
    ax1.set_xlabel('Ground Truth')
    ax1.set_ylabel('Predicted')
    ax1.set_title(f'{task_name}: {model_name}', fontweight='bold')
    ax1.legend()
    
    # Learning curves
    if history:
        ax2 = plt.subplot(1, 2, 2)
        epochs = range(1, len(history['train_loss']) + 1)
        ax2.plot(epochs, history['train_loss'], label='Train Loss', color='#31a354', lw=2)
        ax2.plot(epochs, history['val_loss'], label='Val Loss', color='#e6550d', lw=2)
        ax2.set_xlabel('Epochs')
        ax2.set_ylabel('MSE Loss')
        ax2.set_title('Learning Curves', fontweight='bold')
        ax2.legend()
    
    plt.tight_layout()
    plt.savefig(f"{model_name}_{task_name}.png", dpi=300, bbox_inches='tight')
    plt.show()

print("[ok] Visualization functions defined")

# ==========================================
# Task 2: Session-Level Cybersickness Prediction
# ==========================================

**Target**: SSQ Total Score (0-209)  
**Models**: Aggregated XGBoost, Sequence-to-One LSTM

## Task 2 Data Loading

In [ ]:
# Load sequence data
X_seq = np.load('../sample_data/X_sequences_sample.npy')  # [N, 16, 12]
y_ssq = np.load('../sample_data/y_labels_sample.npy')  # [N]
seq_lengths = np.load('../sample_data/seq_lengths_sample.npy')  # [N]

with open('../sample_data/session_info_sample.pkl', 'rb') as f:
    session_info = pickle.load(f)

# SSQ normalization (prevent gradient explosion)
y_ssq_min, y_ssq_max = y_ssq.min(), y_ssq.max()
y_ssq_norm = (y_ssq - y_ssq_min) / (y_ssq_max - y_ssq_min)

# Build participant-to-session index mapping
p2s = {}
for idx, info in enumerate(session_info):
    p = info['participant']
    if p not in p2s:
        p2s[p] = []
    p2s[p].append(idx)

print(f'[Task 2] Loaded {len(X_seq)} sessions')
print(f'         SSQ range: {y_ssq.min():.1f} - {y_ssq.max():.1f}')
print(f'         SSQ normalized to [0, 1]')

## Task 2 Model Definitions

In [ ]:
# Sequence-to-One LSTM
class SequenceToOneLSTM(nn.Module):
    def __init__(self, input_size=12, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, 2, batch_first=True, dropout=0.3)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1))
    
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        return self.head(h_n[-1]).squeeze(-1)

print("[✓] Task 2 models defined")

## Task 2 Training & Evaluation

In [ ]:
# Aggregated XGBoost LOSO-CV
def run_agg_xgb_task2():
    y_true_all, y_pred_all = [], []
    for test_p in tqdm(PARTICIPANTS, desc='Agg-XGB'):
        train_idx = [i for i, info in enumerate(session_info) if info['participant'] != test_p]
        test_idx = [i for i, info in enumerate(session_info) if info['participant'] == test_p]
        
        # Aggregate: mean + std across time
        X_tr = np.hstack([X_seq[train_idx].mean(axis=1), X_seq[train_idx].std(axis=1)])
        X_te = np.hstack([X_seq[test_idx].mean(axis=1), X_seq[test_idx].std(axis=1)])
        y_tr, y_te = y_ssq[train_idx], y_ssq[test_idx]
        
        X_tr, X_te = fit_transform_X(X_tr, X_te)
        
        model = xgb.XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=SEED)
        model.fit(X_tr, y_tr)
        y_pred_all.extend(model.predict(X_te))
        y_true_all.extend(y_te)
    
    y_true_all, y_pred_all = np.array(y_true_all), np.array(y_pred_all)
    metrics = eval_regression(y_true_all, y_pred_all)
    return metrics, y_true_all, y_pred_all

agg_xgb_result, agg_xgb_y_true, agg_xgb_y_pred = run_agg_xgb_task2()
print(f"[Agg-XGB] RMSE={agg_xgb_result['RMSE']:.3f}, MAE={agg_xgb_result['MAE']:.3f}, r={agg_xgb_result['r']:.3f}")

In [ ]:
# Sequence-to-One LSTM LOSO-CV
def run_lstm_task2():
    y_true_all, y_pred_all = [], []
    last_history = None
    for test_p in tqdm(PARTICIPANTS, desc='LSTM'):
        train_idx = [i for i, info in enumerate(session_info) if info['participant'] != test_p]
        test_idx = [i for i, info in enumerate(session_info) if info['participant'] == test_p]
        
        X_tr, y_tr = X_seq[train_idx], y_ssq_norm[train_idx]
        X_te, y_te = X_seq[test_idx], y_ssq_norm[test_idx]
        len_tr, len_te = seq_lengths[train_idx], seq_lengths[test_idx]
        
        X_tr, X_te = fit_transform_X(X_tr.reshape(-1, 12), X_te.reshape(-1, 12))
        X_tr, X_te = X_tr.reshape(-1, 16, 12), X_te.reshape(-1, 16, 12)
        
        model = SequenceToOneLSTM().to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        
        n_val = max(4, int(len(X_tr) * 0.15))
        X_sub, X_val = X_tr[:-n_val], X_tr[-n_val:]
        y_sub, y_val = y_tr[:-n_val], y_tr[-n_val:]
        len_sub, len_val = len_tr[:-n_val], len_tr[-n_val:]
        
        # Explicit dtype conversion to float32 and int64
        loader = DataLoader(TensorDataset(
            torch.tensor(X_sub, dtype=torch.float32).to(DEVICE), 
            torch.tensor(len_sub, dtype=torch.int64),
            torch.tensor(y_sub, dtype=torch.float32).to(DEVICE)), 
            batch_size=32, shuffle=True)
        
        best_val, best_wts, pat_cnt = float('inf'), copy.deepcopy(model.state_dict()), 0
        last_history = {'train_loss': [], 'val_loss': []}
        
        for ep in range(1, 51):
            model.train()
            epoch_loss = 0.0
            for xb, lb, yb in loader:
                opt.zero_grad()
                loss = nn.MSELoss()(model(xb, lb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                epoch_loss += loss.item()
            
            model.eval()
            with torch.no_grad():
                val_loss = nn.MSELoss()(
                    model(torch.tensor(X_val, dtype=torch.float32).to(DEVICE), 
                          torch.tensor(len_val, dtype=torch.int64)), 
                    torch.tensor(y_val, dtype=torch.float32).to(DEVICE)).item()
            
            last_history['train_loss'].append(epoch_loss / len(loader))
            last_history['val_loss'].append(val_loss)
            
            if val_loss < best_val - 1e-4:
                best_val, best_wts, pat_cnt = val_loss, copy.deepcopy(model.state_dict()), 0
            else:
                pat_cnt += 1
                if pat_cnt >= 10: break
        
        model.load_state_dict(best_wts)
        model.eval()
        with torch.no_grad():
            pred_norm = model(torch.tensor(X_te, dtype=torch.float32).to(DEVICE), 
                            torch.tensor(len_te, dtype=torch.int64)).cpu().numpy()
        
        pred = pred_norm * (y_ssq_max - y_ssq_min) + y_ssq_min
        y_te_orig = y_ssq[test_idx]
        
        y_pred_all.extend(pred)
        y_true_all.extend(y_te_orig)
    
    y_true_all, y_pred_all = np.array(y_true_all), np.array(y_pred_all)
    metrics = eval_regression(y_true_all, y_pred_all)
    return metrics, y_true_all, y_pred_all, last_history

lstm_result, lstm_y_true, lstm_y_pred, lstm_history = run_lstm_task2()
print(f"[LSTM] RMSE={lstm_result['RMSE']:.3f}, MAE={lstm_result['MAE']:.3f}, r={lstm_result['r']:.3f}")

In [ ]:
# Task 2 results summary
task2_results = pd.DataFrame({
    'Model': ['Aggregated XGBoost', 'Sequence-to-One LSTM'],
    'RMSE': [agg_xgb_result['RMSE'], lstm_result['RMSE']],
    'MAE': [agg_xgb_result['MAE'], lstm_result['MAE']],
    'r': [agg_xgb_result['r'], lstm_result['r']]
})
print("\n[Task 2: Session-Level Prediction - Regression Results]")
print(task2_results.to_string(index=False))

# ==========================================
# Visualization (Publication-Quality Plots)
# ==========================================

In [ ]:
# Training loss curves
def plot_learning_curves(history, model_name):
    if history is None: return
    plt.figure(figsize=(6, 5))
    epochs = range(1, len(history['train_loss']) + 1)
    plt.plot(epochs, history['train_loss'], label='Train', color='#31a354', lw=2.5)
    plt.plot(epochs, history['val_loss'], label='Val', color='#e6550d', lw=2.5)
    plt.title(f'{model_name} Learning Curves', fontweight='bold')
    plt.xlabel('Epochs')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'fig_{model_name}_loss.pdf', dpi=300)
    plt.show()

plot_learning_curves(gru_history, "Task1_GRU")
plot_learning_curves(fusion_history, "Task1_Fusion")
plot_learning_curves(lstm_history, "Task2_LSTM")

In [ ]:
# Prediction scatter plots
def plot_prediction_scatter(y_true, y_pred, model_name, task_name):
    plt.figure(figsize=(6, 6))
    sns.regplot(x=y_true, y=y_pred, 
                scatter_kws={'alpha':0.6, 's':30, 'color':'#2b8cbe'}, 
                line_kws={'color':'#de2d26', 'linewidth':2})
    min_val, max_val = min(y_true), max(y_true)
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', lw=1.5, label='Perfect Fit')
    plt.title(f'{model_name}\n{task_name}', fontweight='bold')
    plt.xlabel('Ground Truth')
    plt.ylabel('Predicted')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'fig_{model_name}_scatter.pdf', dpi=300)
    plt.show()

plot_prediction_scatter(gru_y_true, gru_y_pred, "GRU", "Task 1: Minute-Level")
plot_prediction_scatter(fusion_y_true, fusion_y_pred, "Fusion", "Task 1: Minute-Level")
plot_prediction_scatter(lstm_y_true, lstm_y_pred, "LSTM", "Task 2: Session-Level")

In [ ]:
print('=' * 60)
print('Dual-Task Benchmark Complete')
print('=' * 60)
print('\nKey design choices:')
print('  1. Sequential validation split (prevents temporal leakage)')
print('  2. Task 2 SSQ normalization (prevents gradient explosion)')
print('  3. Early stopping (all deep learning models)')
print('  4. Publication-quality visualizations')
print('\nGenerated files:')
print('  - fig_Task1_GRU_loss.pdf')
print('  - fig_Task1_Fusion_loss.pdf')
print('  - fig_Task2_LSTM_loss.pdf')
print('  - fig_GRU_scatter.pdf')
print('  - fig_Fusion_scatter.pdf')
print('  - fig_LSTM_scatter.pdf')
print('=' * 60)

# ==========================================
# Extended Visualization
# ==========================================

In [ ]:
# Chart 1: Top XGBoost Feature Importances
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['font.family'] = 'sans-serif'

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
importance = xgb_model.feature_importances_
top_n = 15
idx = np.argsort(importance)[-top_n:]

plt.figure(figsize=(8, 6))
sns.barplot(x=importance[idx], y=np.array(FEAT_COLS)[idx], palette="viridis")
plt.title('Top XGBoost Feature Importances (Task 1)', fontweight='bold', fontsize=14)
plt.xlabel('Importance Score (Gain)', fontweight='bold')
plt.ylabel('Physiological & Eye Features', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_xgb_importance.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('[ok] Chart 1 saved: fig_xgb_importance.pdf')

In [ ]:
# Chart 1b: Linear Regression Coefficients (Naive Baseline)
top_n = 15
idx = np.argsort(lr_coefs)[-top_n:]

plt.figure(figsize=(8, 6))
sns.barplot(x=lr_coefs[idx], y=np.array(FEAT_COLS)[idx], palette="rocket")
plt.title('Top Linear Regression Coefficients (Task 1 Baseline)', fontweight='bold', fontsize=14)
plt.xlabel('Absolute Coefficient Value', fontweight='bold')
plt.ylabel('Physiological & Eye Features', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_lr_coefficients.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("[✓] Chart 1b saved: fig_lr_coefficients.pdf")

In [ ]:
# Chart 1c: Linear Regression Prediction Scatter
plt.figure(figsize=(6, 6))
sns.regplot(x=lr_y_true, y=lr_y_pred, 
            scatter_kws={'alpha':0.6, 's':30, 'color':'#756bb1'}, 
            line_kws={'color':'#de2d26', 'linewidth':2, 'label':'Linear Fit'})

# Y=X diagonal (perfect prediction)
min_val, max_val = min(lr_y_true), max(lr_y_true)
plt.plot([min_val, max_val], [min_val, max_val], 'k--', lw=1.5, label='Perfect Fit (Y=X)')

plt.title('Linear Regression: Prediction vs Ground Truth\n(Task 1 Baseline)', fontweight='bold', fontsize=14)
plt.xlabel('Ground Truth Score', fontweight='bold')
plt.ylabel('Predicted Score', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('fig_lr_scatter.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('[ok] Chart 1c saved: fig_lr_scatter.pdf')

In [ ]:
# Chart 2: Single-Session Continuous Tracking
# Extract F01 Session 2 (order==2, max score 8.0)
test_p = 'F01'
te_full = loso_split(df, test_p)[1]
te_session2 = te_full[te_full['order'] == 2].sort_values('time').reset_index(drop=True)

time_steps = np.arange(len(te_session2))

# Locate F01 Session 2 in the full prediction array
f01_idx = PARTICIPANTS.index(test_p)
offset = 0
for p in PARTICIPANTS[:f01_idx]:
    offset += len(df[df['participant'] == p])

f01_data = df[df['participant'] == test_p].sort_values(['order', 'time'])
session2_start = len(f01_data[f01_data['order'] < 2])
session2_end = session2_start + len(te_session2)

# Fusion predictions (full session)
fusion_pred_session2 = fusion_y_pred[offset + session2_start : offset + session2_end]

# GRU predictions (window=5, first 4 minutes unavailable)
X_gru_session2, y_gru_session2 = make_windows(te_session2)
gru_offset = 0
for p in PARTICIPANTS[:f01_idx]:
    p_data = df[df['participant'] == p]
    _, y_gru_p = make_windows(p_data)
    gru_offset += len(y_gru_p)
f01_sessions_before = f01_data[f01_data['order'] < 2]
for order in sorted(f01_sessions_before['order'].unique()):
    session_data = f01_sessions_before[f01_sessions_before['order'] == order]
    _, y_gru_s = make_windows(session_data)
    gru_offset += len(y_gru_s)

gru_pred_session2 = gru_y_pred[gru_offset:gru_offset+len(y_gru_session2)]

plt.figure(figsize=(12, 5))
plt.plot(time_steps, te_session2['score'].values, 'k-', lw=2.5,
         label='Ground Truth', alpha=0.8, marker='o', markersize=4)
plt.plot(time_steps, fusion_pred_session2, color='#e6550d', linestyle='-.', lw=2.5,
         label='Multimodal Fusion', alpha=0.9, marker='s', markersize=3)
plt.plot(time_steps[4:], gru_pred_session2, color='#3182bd', linestyle='--', lw=2.5,
         label='GRU (window=5)', alpha=0.9, marker='^', markersize=3)
plt.axhline(y=SICK_THRESHOLD, color='gray', linestyle=':', lw=2, label=f'Sick Threshold ({SICK_THRESHOLD})')
plt.xlabel('Time (Minutes)', fontsize=13, fontweight='bold')
plt.ylabel('Cybersickness Score (0-8)', fontsize=13, fontweight='bold')
plt.title(f'Continuous Tracking: Participant {test_p}, Session 2', fontweight='bold', fontsize=15)
plt.legend(loc='upper left', frameon=True, shadow=True, fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.xlim(-0.5, len(time_steps)-0.5)
plt.ylim(-0.5, 8.5)
plt.tight_layout()
plt.savefig('fig_tracking_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('[ok] Chart 2 saved: fig_tracking_comparison.pdf')

In [ ]:
# Chart 3: Task 2 SSQ Prediction Error Distribution
agg_errors = np.abs(agg_xgb_y_true - agg_xgb_y_pred)
lstm_errors = np.abs(lstm_y_true - lstm_y_pred)

error_df = pd.DataFrame({
    'Absolute Error': np.concatenate([agg_errors, lstm_errors]),
    'Model': ['Agg-XGBoost']*len(agg_errors) + ['LSTM']*len(lstm_errors)
})

plt.figure(figsize=(8, 6))
sns.violinplot(data=error_df, x='Model', y='Absolute Error', palette=['#3182bd', '#e6550d'])
plt.title('SSQ Prediction Error Distribution (Task 2)', fontweight='bold', fontsize=14)
plt.ylabel('Absolute Error (SSQ Score)', fontweight='bold')
plt.xlabel('Model', fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig_ssq_error_violin.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("[✓] Chart 3 saved: fig_ssq_error_violin.pdf")

In [ ]:
# Chart 4: Joint Learning Curves (1x3 grid)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_data = [
    (gru_history, 'Task 1: GRU', axes[0]),
    (fusion_history, 'Task 1: Multimodal Fusion', axes[1]),
    (lstm_history, 'Task 2: LSTM', axes[2])
]

for history, title, ax in models_data:
    epochs = range(1, len(history['train_loss']) + 1)
    ax.plot(epochs, history['train_loss'], label='Train Loss', color='#31a354', lw=2.5, marker='o', markersize=3)
    ax.plot(epochs, history['val_loss'], label='Validation Loss', color='#e6550d', lw=2.5, marker='s', markersize=3)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_xlabel('Epochs', fontweight='bold')
    ax.set_ylabel('MSE Loss', fontweight='bold')
    ax.legend(frameon=True, shadow=True)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig_learning_curves.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('[ok] Chart 4 saved: fig_learning_curves.pdf')